In [ ]:


from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install rasterio

In [ ]:
import rasterio
import numpy as np
import pandas as pd
from glob import glob
from sklearn.utils import shuffle
import os
from time import time
import sys
from rasterio.windows import Window
from rasterio.transform import xy as transform_xy
from PIL import Image

In [ ]:


FILE_TO_PROCESS = "/content/drive/MyDrive/GEE S2_Cloud_2023_MAX_Data/S2_49_TRAINING.tif"

OUTPUT_DIR = "/content/drive/MyDrive/GEE S2_Training_Data/Dask_Split_by_Image"
OUTPUT_FILENAME = "S2_49_TRAINING_data.csv"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

RGB_IMAGES_DIR = os.path.join(OUTPUT_DIR, "RGB_Tiles")
RGB_WITH_CLOUD_MASK_DIR = os.path.join(OUTPUT_DIR, "RGB_Cloud_Mask_Tiles")


os.makedirs(RGB_IMAGES_DIR, exist_ok=True)
os.makedirs(RGB_WITH_CLOUD_MASK_DIR, exist_ok=True)


TILE_SIZE = 512
CLOUD_COVER_MIN = 0
CLOUD_COVER_MAX = 0.75


NAN_THRESHOLD = 0.30
MIN_VALID_PIXELS = 100

MAX_SAMPLES_PER_CLASS = 1500000
MAX_CLEAR_SAMPLES_PER_TILE = 10000


BAND_MAP = {
    'B01': 1, 'B02': 2, 'B03': 3, 'B04': 4, 'B05': 5, 'B06': 6,
    'B07': 7, 'B08': 8, 'B8A': 9, 'B09': 10, 'B11': 11, 'B12': 12,
    'SCL': 13
}
CLOUD_SCL_CODES = [8,9]


RGB_BANDS = {'R': 'B04', 'G': 'B03', 'B': 'B02'}


CLOUD_MASK_COLOR = [255, 0, 0]
CLOUD_MASK_ALPHA = 0.5

FEATURE_NAMES = ['X_Coord', 'Y_Coord'] + list(BAND_MAP.keys())[:-1] + [
    'NDVI', 'NDSI', 'NDWI', 'B11_B08_Ratio', 'B02_B04_Ratio',
    'NDMI', 'B8A_B11_Ratio', 'B01_B11_Ratio', 'B05_B04_Ratio', 'B12_B03_Ratio'
]
LABEL_NAME = 'LABEL'


global_tile_counter = 0
global_sample_counter = 0

In [ ]:



def safe_divide(a, b):
    """Phép chia an toàn, tránh chia cho 0 và NaN."""
    with np.errstate(divide='ignore', invalid='ignore'):
        result = np.divide(a, b)
        result[np.isinf(result)] = 0
        result[np.isnan(result)] = 0
        return result


def check_nan_percentage(data_chunk, nan_threshold=NAN_THRESHOLD):
    """
    Kiểm tra tỷ lệ NaN trong dữ liệu.
    """
    if data_chunk is None or data_chunk.size == 0:
        return 1.0, True, None

    if len(data_chunk.shape) == 3:
        important_bands = [1, 2, 3, 7]
        nan_masks = []
        for band_idx in important_bands:
            if band_idx < data_chunk.shape[0]:
                band_data = data_chunk[band_idx]
                nan_mask = np.isnan(band_data) | (band_data == 0)
                nan_masks.append(nan_mask)

        if nan_masks:
            combined_nan_mask = np.logical_or.reduce(nan_masks)
        else:
            combined_nan_mask = np.isnan(data_chunk[0]) | (data_chunk[0] == 0)
    else:
        combined_nan_mask = np.isnan(data_chunk) | (data_chunk == 0)

    total_pixels = combined_nan_mask.size
    nan_pixels = np.sum(combined_nan_mask)
    nan_percentage = nan_pixels / total_pixels if total_pixels > 0 else 1.0

    exceeds_threshold = nan_percentage >= nan_threshold
    valid_mask = ~combined_nan_mask

    return nan_percentage, exceeds_threshold, valid_mask

In [ ]:
def create_rgb_image(bands_data, band_map, rgb_bands, stretch_percentile=2):
    """
    Tạo ảnh RGB từ các dải phổ Sentinel-2.
    """
    try:
        for band_name in rgb_bands.values():
            if band_name not in bands_data:
                print(f"  Thiếu band {band_name} cho RGB")
                return None

        red_band = bands_data[rgb_bands['R']].astype(np.float32)
        green_band = bands_data[rgb_bands['G']].astype(np.float32)
        blue_band = bands_data[rgb_bands['B']].astype(np.float32)

        # Tạo stack RGB
        rgb_stack = np.stack([red_band, green_band, blue_band], axis=-1)

        # Áp dụng clip và stretch histogram
        rgb_stretched = np.zeros_like(rgb_stack, dtype=np.float32)

        for i in range(3):
            band_data = rgb_stack[:, :, i]

            # Loại bỏ NaN và Inf
            band_data = np.nan_to_num(band_data, nan=0.0, posinf=0.0, neginf=0.0)

            # Chỉ tính percentiles trên giá trị > 0
            valid_data = band_data[band_data > 0]
            if len(valid_data) > 100:
                lower = np.percentile(valid_data, stretch_percentile)
                upper = np.percentile(valid_data, 100 - stretch_percentile)
            else:
                lower, upper = 0, np.max(band_data) if np.max(band_data) > 0 else 1

            # Clip và stretch
            if upper > lower:
                band_clipped = np.clip(band_data, lower, upper)
                band_stretched = (band_clipped - lower) / (upper - lower + 1e-10)
            else:
                band_stretched = band_data / (upper + 1e-10) if upper > 0 else band_data

            band_stretched = np.clip(band_stretched, 0, 1)

            # Chuyển sang 0-255
            rgb_stretched[:, :, i] = band_stretched * 255

        return rgb_stretched.astype(np.uint8)

    except Exception as e:
        print(f"  Lỗi tạo ảnh RGB: {e}")
        return None


In [ ]:
def create_rgb_with_cloud_mask(rgb_array, cloud_mask, cloud_color=CLOUD_MASK_COLOR, alpha=CLOUD_MASK_ALPHA):
    try:
        if rgb_array is None or cloud_mask is None:
            return None

        # Tạo bản sao của ảnh RGB
        rgb_with_mask = rgb_array.copy()

        # Tạo mask 3D từ mask 2D
        cloud_mask_3d = np.stack([cloud_mask, cloud_mask, cloud_mask], axis=-1)

        # Tạo mask màu
        cloud_color_array = np.array(cloud_color, dtype=np.uint8)
        cloud_color_mask = np.full_like(rgb_array, cloud_color_array)

        # Áp dụng mask với độ trong suốt
        rgb_with_mask = np.where(
            cloud_mask_3d > 0,
            (rgb_with_mask * (1 - alpha) + cloud_color_mask * alpha).astype(np.uint8),
            rgb_with_mask
        )

        return rgb_with_mask

    except Exception as e:
        print(f" Lỗi tạo ảnh với mask mây: {e}")
        return rgb_array  # Trả về ảnh gốc nếu có lỗi

In [ ]:

def save_rgb_image(rgb_array, tile_id, file_name, output_dir, cloud_coverage, nan_percentage, image_type="rgb"):

    try:
        # Tạo tên file với thông tin đầy đủ
        base_name = os.path.splitext(file_name)[0]

        if image_type == "cloud_mask":
            rgb_filename = (f"{base_name}_tile_{tile_id:04d}_"
                           f"cloud_{cloud_coverage:.2f}_"
                           f"nan_{nan_percentage:.2f}_MASKED.png")
        else:
            rgb_filename = (f"{base_name}_tile_{tile_id:04d}.png")

        rgb_path = os.path.join(output_dir, rgb_filename)

        # Tạo ảnh từ numpy array
        img = Image.fromarray(rgb_array)

        # Lưu ảnh với chất lượng tốt
        img.save(rgb_path, 'PNG', optimize=True, quality=95)

        if image_type == "cloud_mask":
            print(f" Đã lưu ảnh với mask mây: {os.path.basename(rgb_path)}")
        else:
            print(f" Đã lưu ảnh RGB: {os.path.basename(rgb_path)}")

        return rgb_path

    except Exception as e:
        print(f"  Lỗi lưu ảnh: {e}")
        return None


In [ ]:
def sample_tile_data(X_tile, Y_tile, tile_id, max_clear_samples=MAX_CLEAR_SAMPLES_PER_TILE):
    if X_tile is None or len(X_tile) == 0:
        return None, None, None

    # Tách dữ liệu theo nhãn
    cloud_mask = Y_tile == 1
    clear_mask = Y_tile == 0

    X_cloud = X_tile[cloud_mask]
    Y_cloud = Y_tile[cloud_mask]

    X_clear = X_tile[clear_mask]
    Y_clear = Y_tile[clear_mask]

    # Lấy TẤT CẢ pixel có mây
    n_cloud = X_cloud.shape[0]

    # Lấy mẫu không mây (phân bố đều, tối đa max_clear_samples)
    n_clear = X_clear.shape[0]
    n_sample_clear = min(n_clear, max_clear_samples)

    if n_sample_clear > 0:
        # Lấy mẫu phân bố đều
        indices_clear = np.random.choice(n_clear, n_sample_clear, replace=False)
        X_clear_sampled = X_clear[indices_clear]
        Y_clear_sampled = Y_clear[indices_clear]
    else:
        X_clear_sampled = np.array([]).reshape(0, X_tile.shape[1])
        Y_clear_sampled = np.array([])

    # Gộp kết quả
    if n_cloud > 0 and len(X_clear_sampled) > 0:
        X_combined = np.concatenate([X_cloud, X_clear_sampled], axis=0)
        Y_combined = np.concatenate([Y_cloud, Y_clear_sampled], axis=0)
    elif n_cloud > 0:
        X_combined, Y_combined = X_cloud, Y_cloud
    elif len(X_clear_sampled) > 0:
        X_combined, Y_combined = X_clear_sampled, Y_clear_sampled
    else:
        return None, None, None

    tile_ids_combined = np.full(Y_combined.shape, tile_id, dtype=np.int32)

    if len(X_combined) > 0:
        X_shuffled, Y_shuffled, tile_ids_shuffled = shuffle(
            X_combined, Y_combined, tile_ids_combined, random_state=42
        )
        return X_shuffled, Y_shuffled, tile_ids_shuffled
    else:
        return None, None, None


def stratified_sampling_by_scl(X_data, Y_data, max_samples_per_class, tile_ids):
    """Thực hiện lấy mẫu có phân tầng cuối cùng từ tất cả tiles."""

    if X_data is None or len(X_data) == 0:
        return None, None, None

    # Tách dữ liệu theo nhãn
    X_cloud = X_data[Y_data == 1]
    Y_cloud = Y_data[Y_data == 1]
    tile_ids_cloud = tile_ids[Y_data == 1]

    X_clear = X_data[Y_data == 0]
    Y_clear = Y_data[Y_data == 0]
    tile_ids_clear = tile_ids[Y_data == 0]

    n_cloud = X_cloud.shape[0]
    n_clear = X_clear.shape[0]

    if n_cloud == 0 and n_clear == 0:
        return None, None, None

    # Lấy mẫu cho Lớp Mây (Label 1) - lấy tất cả hoặc tối đa max_samples_per_class
    n_sample_cloud = min(n_cloud, max_samples_per_class)
    if n_cloud > 0 and n_sample_cloud > 0:
        indices_cloud = np.random.choice(n_cloud, n_sample_cloud, replace=False)
        X_cloud_sampled = X_cloud[indices_cloud]
        Y_cloud_sampled = Y_cloud[indices_cloud]
        tile_ids_cloud_sampled = tile_ids_cloud[indices_cloud]
    else:
        X_cloud_sampled = np.array([]).reshape(0, X_data.shape[1])
        Y_cloud_sampled = np.array([])
        tile_ids_cloud_sampled = np.array([])

    # Lấy mẫu cho Lớp Không Mây (Label 0) - tối đa max_samples_per_class
    n_sample_clear = min(n_clear, max_samples_per_class)
    if n_clear > 0 and n_sample_clear > 0:
        indices_clear = np.random.choice(n_clear, n_sample_clear, replace=False)
        X_clear_sampled = X_clear[indices_clear]
        Y_clear_sampled = Y_clear[indices_clear]
        tile_ids_clear_sampled = tile_ids_clear[indices_clear]
    else:
        X_clear_sampled = np.array([]).reshape(0, X_data.shape[1])
        Y_clear_sampled = np.array([])
        tile_ids_clear_sampled = np.array([])

    # Gộp kết quả
    if len(X_cloud_sampled) > 0 and len(X_clear_sampled) > 0:
        X_combined = np.concatenate([X_cloud_sampled, X_clear_sampled], axis=0)
        Y_combined = np.concatenate([Y_cloud_sampled, Y_clear_sampled], axis=0)
        tile_ids_combined = np.concatenate([tile_ids_cloud_sampled, tile_ids_clear_sampled], axis=0)
    elif len(X_cloud_sampled) > 0:
        X_combined, Y_combined, tile_ids_combined = X_cloud_sampled, Y_cloud_sampled, tile_ids_cloud_sampled
    elif len(X_clear_sampled) > 0:
        X_combined, Y_combined, tile_ids_combined = X_clear_sampled, Y_clear_sampled, tile_ids_clear_sampled
    else:
        return None, None, None

    # Xáo trộn cuối cùng
    if len(X_combined) > 0:
        X_shuffled, Y_shuffled, tile_ids_shuffled = shuffle(
            X_combined, Y_combined, tile_ids_combined, random_state=42
        )
        return X_shuffled, Y_shuffled, tile_ids_shuffled
    else:
        return None, None, None


def calculate_coordinates_safe(transform, rows, cols, start_row, start_col):
    """
    Tính toán tọa độ một cách an toàn, đảm bảo đầu ra là 2D.
    """
    try:
        # Tạo mảng tọa độ
        x_coords = np.zeros((rows, cols), dtype=np.float64)
        y_coords = np.zeros((rows, cols), dtype=np.float64)

        # Tính tọa độ cho từng pixel
        for i in range(rows):
            for j in range(cols):
                x, y = transform_xy(
                    transform,
                    start_row + i,
                    start_col + j,
                    offset='center'
                )
                x_coords[i, j] = x
                y_coords[i, j] = y

        return x_coords, y_coords

    except Exception as e:
        print(f"    Lỗi tính tọa độ: {e}, dùng tọa độ mặc định")
        # Fallback
        x_res, _, x_min, _, y_res, y_max = transform[:6]
        x_coords = np.zeros((rows, cols), dtype=np.float64)
        y_coords = np.zeros((rows, cols), dtype=np.float64)

        for i in range(rows):
            for j in range(cols):
                x_coords[i, j] = x_min + (start_col + j + 0.5) * x_res
                y_coords[i, j] = y_max - (start_row + i + 0.5) * abs(y_res)

        return x_coords, y_coords


def process_file_and_sample(file_path):
    """
    Chia tệp thành các ô (tiles), Lọc ô theo tỷ lệ mây và NaN,
    tính toán Features, Tọa độ, và Lấy mẫu Cân bằng.
    Lưu ảnh RGB và ảnh RGB với mask mây.
    """
    global global_tile_counter

    list_X_chunks = []
    list_Y_chunks = []
    list_tile_ids = []
    file_name = os.path.basename(file_path)

    # Dictionary lưu thông tin tiles
    tile_info_dict = {}

    # Thống kê
    stats = {
        'total_tiles': 0,
        'cloud_filtered': 0,
        'nan_filtered': 0,
        'valid_tiles': 0,
        'total_samples': 0,
        'images_saved': 0,
        'masked_images_saved': 0,
        'cloud_samples': 0,
        'clear_samples': 0
    }

    try:
        with rasterio.open(file_path) as src:
            print(f"[{file_name}] Kích thước ảnh: {src.width}x{src.height}")
            print(f"[{file_name}] Số band: {src.count}")

            # Tạo danh sách các cửa sổ
            windows = []
            height, width = src.height, src.width

            for row in range(0, height, TILE_SIZE):
                for col in range(0, width, TILE_SIZE):
                    window = Window(col_off=col,
                                   row_off=row,
                                   width=min(TILE_SIZE, width - col),
                                   height=min(TILE_SIZE, height - row))
                    if window.width >= 64 and window.height >= 64:
                        windows.append(window)

            stats['total_tiles'] = len(windows)
            print(f"[{file_name}] Tổng số ô hợp lệ: {stats['total_tiles']}")
            print(f"[{file_name}] Ngưỡng NaN: {NAN_THRESHOLD*100:.0f}%")
            print(f"[{file_name}] Tối đa mẫu không mây/tile: {MAX_CLEAR_SAMPLES_PER_TILE:,}")

            # Lặp qua từng ô
            for idx, window in enumerate(windows):
                stats['total_tiles'] += 1

                try:
                    # 1. ĐỌC DỮ LIỆU VÀ KIỂM TRA NAN
                    full_data_chunk = src.read(window=window)

                    if full_data_chunk is None or full_data_chunk.size == 0:
                        continue

                    # Kiểm tra tỷ lệ NaN
                    nan_percentage, exceeds_nan_threshold, valid_mask = check_nan_percentage(
                        full_data_chunk, NAN_THRESHOLD
                    )

                    if exceeds_nan_threshold:
                        stats['nan_filtered'] += 1
                        if idx % 20 == 0:
                            print(f"  - Ô {idx+1}: Bỏ qua do nhiều NaN ({nan_percentage:.1%})")
                        continue

                    # 2. ĐỌC SCL RIÊNG ĐỂ TÍNH TỶ LỆ MÂY
                    scl_idx = BAND_MAP['SCL'] - 1
                    if scl_idx < full_data_chunk.shape[0]:
                        scl_chunk = full_data_chunk[scl_idx].astype(np.uint8)
                    else:
                        scl_chunk = src.read(BAND_MAP['SCL'], window=window).astype(np.uint8)

                    if len(scl_chunk.shape) != 2:
                        continue

                    rows, cols = scl_chunk.shape

                    # Tính tỷ lệ mây trong ô
                    nan_mask_scl = (scl_chunk == 0) | (scl_chunk == 1)
                    is_cloud_mask = np.isin(scl_chunk, CLOUD_SCL_CODES)
                    total_valid_pixels = np.sum(~nan_mask_scl)

                    if total_valid_pixels < MIN_VALID_PIXELS:
                        continue

                    cloud_coverage = np.sum(is_cloud_mask) / total_valid_pixels

                    # 3. LỌC THEO TỶ LỆ MÂY
                    if not (CLOUD_COVER_MIN <= cloud_coverage <= CLOUD_COVER_MAX):
                        stats['cloud_filtered'] += 1
                        continue

                    stats['valid_tiles'] += 1

                    # 4. CHUẨN BỊ DỮ LIỆU BANDS
                    full_data_chunk = np.nan_to_num(full_data_chunk, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

                    Bands_chunk = {}
                    for name, band_idx in BAND_MAP.items():
                        idx_0based = band_idx - 1
                        if idx_0based < full_data_chunk.shape[0]:
                            Bands_chunk[name] = full_data_chunk[idx_0based, :, :]

                    # 5. TẠO VÀ LƯU ẢNH RGB THƯỜNG
                    rgb_image = create_rgb_image(Bands_chunk, BAND_MAP, RGB_BANDS)

                    if rgb_image is not None and rgb_image.shape[:2] == (rows, cols):
                        global_tile_counter += 1
                        tile_id = global_tile_counter

                        # Lưu ảnh RGB thường
                        rgb_path = save_rgb_image(
                            rgb_image, tile_id, file_name,
                            RGB_IMAGES_DIR, cloud_coverage, nan_percentage, "rgb"
                        )

                        if rgb_path:
                            stats['images_saved'] += 1

                        # 6. TẠO VÀ LƯU ẢNH RGB VỚI MASK MÂY
                        # Tạo mask mây từ SCL
                        cloud_mask = np.isin(scl_chunk, CLOUD_SCL_CODES).astype(np.uint8)

                        # Tạo ảnh RGB với mask mây
                        rgb_with_cloud_mask = create_rgb_with_cloud_mask(rgb_image, cloud_mask)

                        if rgb_with_cloud_mask is not None:
                            # Lưu ảnh với mask mây
                            rgb_masked_path = save_rgb_image(
                                rgb_with_cloud_mask, tile_id, file_name,
                                RGB_WITH_CLOUD_MASK_DIR, cloud_coverage, nan_percentage, "cloud_mask"
                            )

                            if rgb_masked_path:
                                stats['masked_images_saved'] += 1

                        # Lưu thông tin tile
                        tile_info_dict[tile_id] = {
                            'file_name': file_name,
                            'window_idx': idx,
                            'window': window,
                            'cloud_coverage': cloud_coverage,
                            'nan_percentage': nan_percentage,
                            'rgb_path': rgb_path,
                            'rgb_masked_path': rgb_masked_path if 'rgb_masked_path' in locals() else None,
                            'tile_id': tile_id,
                            'rows': rows,
                            'cols': cols,
                            'valid_pixels': total_valid_pixels,
                            'cloud_pixels': np.sum(is_cloud_mask),
                            'clear_pixels': total_valid_pixels - np.sum(is_cloud_mask)
                        }

                        print(f"  - Ô {idx+1}/{len(windows)}: Tile {tile_id}, "
                              f"{rows}x{cols}, mây: {cloud_coverage:.1%} "
                              f"(mây: {tile_info_dict[tile_id]['cloud_pixels']}px, "
                              f"không mây: {tile_info_dict[tile_id]['clear_pixels']}px), "
                              f"NaN: {nan_percentage:.1%}")

                    # 7. TÍNH TOÁN TỌA ĐỘ
                    x_coords_2D, y_coords_2D = calculate_coordinates_safe(
                        src.transform, rows, cols, window.row_off, window.col_off
                    )

                    # 8. TÍNH FEATURES PHÁI SINH
                    try:
                        required_bands = ['B02', 'B03', 'B04', 'B08', 'B11', 'B8A', 'B01', 'B05', 'B12']
                        for band in required_bands:
                            if band not in Bands_chunk:
                                raise KeyError(f"Thiếu band {band}")

                        ndvi = safe_divide(Bands_chunk['B08'] - Bands_chunk['B04'],
                                          Bands_chunk['B08'] + Bands_chunk['B04'])
                        ndsi = safe_divide(Bands_chunk['B03'] - Bands_chunk['B11'],
                                          Bands_chunk['B03'] + Bands_chunk['B11'])
                        ndwi = safe_divide(Bands_chunk['B03'] - Bands_chunk['B08'],
                                          Bands_chunk['B03'] + Bands_chunk['B08'])
                        b11_b08_ratio = safe_divide(Bands_chunk['B11'],
                                                   Bands_chunk['B08'])
                        b02_b04_ratio = safe_divide(Bands_chunk['B02'],
                                                   Bands_chunk['B04'])
                        ndmi = safe_divide(Bands_chunk['B08'] - Bands_chunk['B11'],
                                          Bands_chunk['B08'] + Bands_chunk['B11'])
                        b8a_b11_ratio = safe_divide(Bands_chunk['B8A'],
                                                   Bands_chunk['B11'])
                        b01_b11_ratio = safe_divide(Bands_chunk['B01'],
                                                   Bands_chunk['B11'])
                        b05_b04_ratio = safe_divide(Bands_chunk['B05'],
                                                   Bands_chunk['B04'])
                        b12_b03_ratio = safe_divide(Bands_chunk['B12'],
                                                   Bands_chunk['B03'])

                    except KeyError as e:
                        print(f"     Thiếu band: {e}, bỏ qua tile")
                        continue
                    except Exception as e:
                        print(f"     Lỗi tính features: {e}, bỏ qua tile")
                        continue

                    # 9. TẠO STACK DỮ LIỆU
                    raw_bands_stack = full_data_chunk[:12, :, :]
                    derived_features_stack = np.stack([
                        ndvi, ndsi, ndwi, b11_b08_ratio, b02_b04_ratio,
                        ndmi, b8a_b11_ratio, b01_b11_ratio, b05_b04_ratio, b12_b03_ratio
                    ], axis=0)

                    # 10. TẠO STACK DỮ LIỆU VỚI TỌA ĐỘ
                    x_coords_stack = x_coords_2D[np.newaxis, :, :]
                    y_coords_stack = y_coords_2D[np.newaxis, :, :]

                    X_all_2D = np.concatenate(
                        [x_coords_stack, y_coords_stack, raw_bands_stack, derived_features_stack],
                        axis=0
                    )

                    # Tạo Labels từ SCL
                    Y_labels_2D = np.isin(scl_chunk, CLOUD_SCL_CODES).astype(np.uint8)

                    # Sử dụng valid_mask
                    if valid_mask is not None and valid_mask.shape == (rows, cols):
                        valid_mask_flatten = valid_mask.flatten()
                    else:
                        valid_mask = (full_data_chunk[0, :, :] != 0)
                        valid_mask_flatten = valid_mask.flatten()

                    # LÀM PHẲNG VÀ LỌC
                    X_clean_chunk = X_all_2D.transpose((1, 2, 0)).reshape(-1, len(FEATURE_NAMES))[valid_mask_flatten]
                    Y_clean_chunk = Y_labels_2D.flatten()[valid_mask_flatten]

                    #  MỚI: Lấy mẫu từ tile này theo quy tắc mới
                    if X_clean_chunk.shape[0] > 0:
                        # Áp dụng sampling cho tile này
                        X_tile_sampled, Y_tile_sampled, tile_ids_tile_sampled = sample_tile_data(
                            X_clean_chunk, Y_clean_chunk, tile_id, MAX_CLEAR_SAMPLES_PER_TILE
                        )

                        if X_tile_sampled is not None and len(X_tile_sampled) > 0:
                            list_X_chunks.append(X_tile_sampled)
                            list_Y_chunks.append(Y_tile_sampled)
                            list_tile_ids.append(tile_ids_tile_sampled)

                            # Thống kê
                            stats['total_samples'] += X_tile_sampled.shape[0]
                            stats['cloud_samples'] += np.sum(Y_tile_sampled == 1)
                            stats['clear_samples'] += np.sum(Y_tile_sampled == 0)

                            print(f"     Tile {tile_id}: Lấy {X_tile_sampled.shape[0]:,} mẫu "
                                  f"(mây: {np.sum(Y_tile_sampled == 1):,}, "
                                  f"không mây: {np.sum(Y_tile_sampled == 0):,})")

                except Exception as e:
                    print(f"  - Ô {idx+1} lỗi: {str(e)[:80]}...")
                    continue

            print(f"\n[{file_name}] THỐNG KÊ:")
            print(f"  - Tổng ô: {stats['total_tiles']}")
            print(f"  - Bỏ qua do NaN ≥{NAN_THRESHOLD*100:.0f}%: {stats['nan_filtered']}")
            print(f"  - Bỏ qua do mây không đạt: {stats['cloud_filtered']}")
            print(f"  - Ô hợp lệ: {stats['valid_tiles']}")
            print(f"  - Tổng mẫu đã lấy: {stats['total_samples']:,}")
            print(f"  - Mẫu mây: {stats['cloud_samples']:,}")
            print(f"  - Mẫu không mây: {stats['clear_samples']:,}")
            print(f"  - Ảnh RGB đã lưu: {stats['images_saved']}")
            print(f"  - Ảnh với mask mây đã lưu: {stats['masked_images_saved']}")

            if not list_X_chunks:
                print(f"[{file_name}] KHÔNG CÓ Ô NÀO THỎA MÃN TẤT CẢ ĐIỀU KIỆN")
                return None, None, None, None

            # Gộp tất cả các mẫu hợp lệ từ tất cả tiles
            X_all = np.concatenate(list_X_chunks, axis=0)
            Y_all = np.concatenate(list_Y_chunks, axis=0)
            tile_ids_all = np.concatenate(list_tile_ids, axis=0)

            print(f"[{file_name}] ĐÃ XỬ LÝ: {stats['valid_tiles']} ô")
            print(f"[{file_name}] TỔNG MẪU: {X_all.shape[0]:,} mẫu "
                  f"(mây: {np.sum(Y_all == 1):,}, không mây: {np.sum(Y_all == 0):,})")
            print(f"[{file_name}] ĐÃ LƯU: {stats['images_saved']} ảnh RGB, {stats['masked_images_saved']} ảnh với mask mây")

            # LẤY MẪU CÂN BẰNG CUỐI CÙNG (toàn bộ dataset)
            X_sampled, Y_sampled, tile_ids_sampled = stratified_sampling_by_scl(
                X_all, Y_all, MAX_SAMPLES_PER_CLASS, tile_ids_all
            )

            return X_sampled, Y_sampled, tile_ids_sampled, tile_info_dict

    except Exception as e:
        print(f"[{file_name}] LỖI XỬ LÝ CHÍNH: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None, None


In [ ]:
def add_file_to_training_set(file_path, output_path):
    """Xử lý một tệp cụ thể, thêm cột Image_ID, và ghi thêm dữ liệu vào tệp CSV."""
    global global_sample_counter

    file_name = os.path.basename(file_path)

    print(f"BẮT ĐẦU XỬ LÝ TỆP ĐƠN LẺ: {file_name}")
    print(f"Kích thước TILE: {TILE_SIZE}x{TILE_SIZE}")
    print(f"Điều kiện mây: {CLOUD_COVER_MIN*100:.0f}% - {CLOUD_COVER_MAX*100:.0f}%")
    print(f"Ngưỡng NaN: {NAN_THRESHOLD*100:.0f}% (bỏ qua nếu ≥)")
    print(f"Tối đa mẫu không mây/tile: {MAX_CLEAR_SAMPLES_PER_TILE:,}")
    print(f"Thư mục ảnh RGB: {RGB_IMAGES_DIR}")
    print(f"Thư mục ảnh RGB với mask mây: {RGB_WITH_CLOUD_MASK_DIR}")

    start_time = time()

    if not os.path.exists(os.path.dirname(output_path)):
        os.makedirs(os.path.dirname(output_path))

    X_sampled, Y_sampled, tile_ids_sampled, tile_info_dict = process_file_and_sample(file_path)

    if X_sampled is not None and len(X_sampled) > 0:

        num_samples = X_sampled.shape[0]
        image_id_col = [file_name] * num_samples

        # Tạo DataFrame
        df_batch = pd.DataFrame(X_sampled, columns=FEATURE_NAMES)
        df_batch[LABEL_NAME] = Y_sampled
        df_batch['Image_ID'] = image_id_col
        df_batch['Tile_ID'] = tile_ids_sampled

        # Thêm thông tin từ tile_info_dict
        cloud_coverages = []
        nan_percentages = []
        cloud_pixels_list = []
        clear_pixels_list = []

        for tile_id in tile_ids_sampled:
            if tile_id in tile_info_dict:
                info = tile_info_dict[tile_id]
                cloud_coverages.append(info['cloud_coverage'])
                nan_percentages.append(info['nan_percentage'])
                cloud_pixels_list.append(info['cloud_pixels'])
                clear_pixels_list.append(info['clear_pixels'])
            else:
                cloud_coverages.append(0.0)
                nan_percentages.append(0.0)
                cloud_pixels_list.append(0)
                clear_pixels_list.append(0)

        df_batch['Tile_Cloud_Coverage'] = cloud_coverages
        df_batch['Tile_Nan_Percentage'] = nan_percentages
        df_batch['Tile_Cloud_Pixels'] = cloud_pixels_list
        df_batch['Tile_Clear_Pixels'] = clear_pixels_list

        # Thêm thông tin sampling
        df_batch['Sample_Type'] = 'cloud'
        df_batch.loc[df_batch[LABEL_NAME] == 0, 'Sample_Type'] = 'clear_sampled'

        # Tạo Sample_ID duy nhất
        start_idx = global_sample_counter
        sample_ids = np.arange(start_idx, start_idx + num_samples)
        df_batch['Sample_ID'] = sample_ids
        global_sample_counter += num_samples

        # Xác định mode ghi
        if not os.path.exists(output_path):
            write_mode = 'w'
            write_header = True
            print(f"Tạo tệp CSV mới: {os.path.basename(output_path)}")
        else:
            write_mode = 'a'
            write_header = False
            print(f"Ghi thêm vào tệp CSV đã tồn tại: {os.path.basename(output_path)}")

        try:
            # Ghi CSV
            df_batch.to_csv(output_path,
                           index=False,
                           mode=write_mode,
                           header=write_header,
                           float_format='%.6f')

            # Lưu thông tin tiles
            if tile_info_dict:
                tile_info_path = os.path.join(OUTPUT_DIR, f"{os.path.splitext(file_name)[0]}_tile_info.csv")
                tile_info_list = []
                for tile_id, info in tile_info_dict.items():
                    tile_info_list.append({
                        'Tile_ID': tile_id,
                        'File_Name': info['file_name'],
                        'Window_Idx': info['window_idx'],
                        'Rows': info['rows'],
                        'Cols': info['cols'],
                        'Cloud_Coverage': info['cloud_coverage'],
                        'Nan_Percentage': info['nan_percentage'],
                        'Valid_Pixels': info['valid_pixels'],
                        'Cloud_Pixels': info['cloud_pixels'],
                        'Clear_Pixels': info['clear_pixels'],
                        'RGB_Path': info['rgb_path'],
                        'RGB_Masked_Path': info['rgb_masked_path'],
                        'Row_Off': info['window'].row_off,
                        'Col_Off': info['window'].col_off,
                        'Width': info['window'].width,
                        'Height': info['window'].height
                    })

                tile_info_df = pd.DataFrame(tile_info_list)
                tile_info_df.to_csv(tile_info_path, index=False)
                print(f"Đã lưu thông tin tiles: {tile_info_path}")

            # Thống kê cuối cùng
            current_samples = X_sampled.shape[0]
            current_cloud = np.sum(Y_sampled)
            processing_time = time() - start_time

            print(f"HOÀN TẤT XỬ LÝ: {file_name}")
            print(f"   - Số mẫu được ghi: {current_samples:,}")
            print(f"   - Mẫu Mây (Label 1): {current_cloud:,}")
            print(f"   - Mẫu Không Mây (Label 0): {current_samples - current_cloud:,}")
            print(f"   - Số tiles đã lưu ảnh: {len(tile_info_dict)}")
            print(f"   - Ảnh RGB: {RGB_IMAGES_DIR}")
            print(f"   - Ảnh với mask mây: {RGB_WITH_CLOUD_MASK_DIR}")
            print(f"   - Tổng thời gian: {processing_time:.2f} giây")
            print(f"   - Tốc độ: {current_samples/processing_time:.0f} mẫu/giây")
            if current_samples > 0:
                cloud_percent = (current_cloud / current_samples * 100)
                print(f"   - Tỷ lệ Mây/Tổng: {cloud_percent:.2f}%")
            print(f"   - Sample_ID từ: {start_idx:,} đến {start_idx + num_samples - 1:,}")

        except Exception as e:
            print(f"LỖI GHI DỮ LIỆU VÀO CSV: {e}")
            import traceback
            traceback.print_exc()

    else:
        print(f"[{file_name}] BỎ QUA: Không có mẫu hợp lệ sau khi lọc.")

In [ ]:
if __name__ == '__main__':
    # Tạo thư mục nếu chưa tồn tại
    os.makedirs(RGB_IMAGES_DIR, exist_ok=True)
    os.makedirs(RGB_WITH_CLOUD_MASK_DIR, exist_ok=True)

    print(" KHỞI CHẠY CHƯƠNG TRÌNH XỬ LÝ ẢNH SENTINEL-2")
    print(f"File đầu vào: {FILE_TO_PROCESS}")
    print(f"Thư mục ảnh RGB: {RGB_IMAGES_DIR}")
    print(f"Thư mục ảnh RGB với mask mây: {RGB_WITH_CLOUD_MASK_DIR}")
    print(f"Thư mục lưu dữ liệu: {OUTPUT_DIR}")
    print(f"File CSV đầu ra: {OUTPUT_PATH}")
    print(f"Kích thước tile: {TILE_SIZE}x{TILE_SIZE}")
    print(f"Ngưỡng lọc NaN: {NAN_THRESHOLD*100:.0f}%")
    print(f"Điều kiện mây: {CLOUD_COVER_MIN*100:.0f}% - {CLOUD_COVER_MAX*100:.0f}%")
    print(f"Tối đa mẫu không mây/tile: {MAX_CLEAR_SAMPLES_PER_TILE:,}")
    print(f"Chiến lược sampling: Lấy TẤT CẢ pixel mây + tối đa {MAX_CLEAR_SAMPLES_PER_TILE:,} pixel không mây/tile")
    print(f"Màu mask mây: Đỏ (RGB: {CLOUD_MASK_COLOR})")
    print(f"Độ trong suốt mask: {CLOUD_MASK_ALPHA*100:.0f}%")

    add_file_to_training_set(FILE_TO_PROCESS, OUTPUT_PATH)